<a href="https://colab.research.google.com/github/Raheel1964/Data-Analysis/blob/main/Agri_trade_agent_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Define a helper function to load and clean each ITC text file
def load_and_clean_itc_data(file_path, chapter_name):
    # Read the tab-separated text file
    df = pd.read_csv(file_path, sep='\t', dtype={'﻿Product Code': str})

    # Rename the first column to fix the hidden BOM character issue often found in ITC txt files
    df.rename(columns={df.columns[0]: 'HS_Code'}, inplace=True)

    # Standardize and clean the notoriously long ITC column names
    column_mapping = {
        "Product Label": "Product_Description",
        "Pakistan's exports to world-Value in 2024, USD thousand": "World_Import_Value_kUSD",
        "Pakistan's exports to world-Annual growth in value between 2020-2024, %, p.a.": "World_Annual_Growth_pct",
        "Pakistan's exports to world-Share in world imports, %": "World_Share_pct",
        "Select your indicators-Value in 2024, USD thousand": "Pak_Export_Value_kUSD",
        "Select your indicators-Annual growth in value between 2020-2024, %, p.a.": "Pak_Export_Growth_pct",
        "Select your indicators-Share in world exports, %": "Pak_World_Share_pct",
        "Select your indicators-Quantity exported in 2024": "Pak_Export_Qty",
        "Select your indicators-Quantity unit": "Pak_Qty_Unit",
        "Select your indicators-Unit value (USD/unit)": "Pak_Unit_Value_USD"
    }

    # Apply renaming, ignoring columns we don't strictly need or empty spacer columns
    df = df.rename(columns=column_mapping)

    # Keep only the mapped columns that exist in the dataframe
    cols_to_keep = ['HS_Code', 'Product_Description', 'World_Import_Value_kUSD',
                    'World_Annual_Growth_pct', 'Pak_Export_Value_kUSD', 'Pak_Export_Growth_pct',
                    'Pak_World_Share_pct', 'Pak_Export_Qty', 'Pak_Qty_Unit', 'Pak_Unit_Value_USD']

    df = df[[col for col in cols_to_keep if col in df.columns]]

    # Add a column for the Chapter grouping
    df.insert(1, 'Chapter', chapter_name)

    # Clean up 'No quantity' strings and fill NaN/Blanks with 0
    df.replace('No quantity', 0, inplace=True)
    df.fillna(0, inplace=True)

    # Convert numeric columns to actual float/int types
    numeric_cols = ['World_Import_Value_kUSD', 'Pak_Export_Value_kUSD', 'Pak_Export_Qty', 'Pak_Unit_Value_USD']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    return df

# 2. Load the files (Assuming they are uploaded to your local directory or Colab)
# Replace the filenames with your actual file paths
df_ch06 = load_and_clean_itc_data('Trade_Map_-chapter_6.txt', 'Ch 06 - Live Plants')
df_ch07 = load_and_clean_itc_data('Trade_Map_Chapter_7_2024.txt', 'Ch 07 - Vegetables')
df_ch08 = load_and_clean_itc_data('Trade_Map_Chapter_8_2024.txt', 'Ch 08 - Fruits & Nuts')
df_ch20 = load_and_clean_itc_data('Trade_Map_-Chapter_20_2024.txt', 'Ch 20 - Preserved Foods')

# 3. Combine them all into one Master DataFrame
master_horti_df = pd.concat([df_ch06, df_ch07, df_ch08, df_ch20], ignore_index=True)

# 4. Preview the cleaned quantitative engine!
print("Data Successfully Ingested! Total Rows:", len(master_horti_df))
display(master_horti_df.head())

Data Successfully Ingested! Total Rows: 41


/tmp/ipython-input-1345261466.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('No quantity', 0, inplace=True)


,HS_Code,Chapter,Product_Description,World_Import_Value_kUSD,World_Annual_Growth_pct,Pak_Export_Value_kUSD,Pak_Export_Growth_pct,Pak_World_Share_pct,Pak_Export_Qty,Pak_Qty_Unit,Pak_Unit_Value_USD
0,601,Ch 06 - Live Plants,"Bulbs, tubers, tuberous roots, corms, crowns a...",1985021,2,1046,32.0,0,517117.0,0,0.0
1,602,Ch 06 - Live Plants,"Live plants incl. their roots, cuttings and sl...",10429494,2,302,-7.0,0,249658.0,0,0.0
2,603,Ch 06 - Live Plants,Cut flowers and flower buds of a kind suitable...,10493492,5,1761,1.0,0,734.0,0,0.0
3,604,Ch 06 - Live Plants,"Foliage, branches and other parts of plants, w...",1425151,2,280,12.0,0,3677.0,0,0.0
4,701,Ch 07 - Vegetables,"Potatoes, fresh or chilled",7646059,15,141324,18.0,2,752882.0,0,0.0


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import google.generativeai as genai

# --- 1. PAGE SETUP ---
st.set_page_config(page_title="Pak-Agri Trade Agent MVP", layout="wide")
st.title("🇵🇰 Pakistan Agri-Trade AI Agent (MVP)")
st.markdown("**Powered by ITC Trade Map & Gemini 1.5 Pro (World Bank/PHDEC Framework)**")

# --- 2. LOAD QUANTITATIVE DATA (ITC) ---
@st.cache_data
def load_data():
    # Helper function from our previous step
    def load_and_clean(file_path, chapter_name):
        try:
            df = pd.read_csv(file_path, sep='\t', dtype={'﻿Product Code': str})
            df.rename(columns={df.columns[0]: 'HS_Code'}, inplace=True)
            col_map = {
                "Product Label": "Product_Description",
                "Select your indicators-Value in 2024, USD thousand": "Pak_Export_Value_kUSD",
                "Select your indicators-Quantity exported in 2024": "Pak_Export_Qty",
                "Select your indicators-Unit value (USD/unit)": "Pak_Unit_Value_USD"
            }
            df = df.rename(columns=col_map)
            cols_to_keep = ['HS_Code', 'Product_Description', 'Pak_Export_Value_kUSD', 'Pak_Export_Qty', 'Pak_Unit_Value_USD']
            df = df[[col for col in cols_to_keep if col in df.columns]]
            df.insert(1, 'Chapter', chapter_name)
            df.replace('No quantity', 0, inplace=True)
            df.fillna(0, inplace=True)
            for col in ['Pak_Export_Value_kUSD', 'Pak_Export_Qty', 'Pak_Unit_Value_USD']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            return df
        except FileNotFoundError:
            return pd.DataFrame() # Return empty if file missing during Colab run

    # Load all chapters
    ch06 = load_and_clean('Trade_Map_-chapter_6.txt', 'Ch 06')
    ch07 = load_and_clean('Trade_Map_Chapter_7_2024.txt', 'Ch 07')
    ch08 = load_and_clean('Trade_Map_Chapter_8_2024.txt', 'Ch 08')
    ch20 = load_and_clean('Trade_Map_-Chapter_20_2024.txt', 'Ch 20')

    return pd.concat([ch06, ch07, ch08, ch20], ignore_index=True)

df = load_data()

# --- 3. SIDEBAR: COMMODITY SELECTION & API KEY ---
st.sidebar.header("Agent Configuration")
api_key = st.sidebar.text_input("Enter Gemini API Key", type="password")

if not df.empty:
    st.sidebar.subheader("Select Commodity (HS Code)")
    # Create a dropdown of HS Codes + Descriptions
    df['Display_Name'] = df['HS_Code'].astype(str) + " - " + df['Product_Description'].str[:40] + "..."
    selected_item = st.sidebar.selectbox("Choose Target:", df['Display_Name'])

    # Extract selected row
    target_data = df[df['Display_Name'] == selected_item].iloc[0]
else:
    st.error("Data files not found. Ensure the .txt files are in the Colab directory.")
    st.stop()

# --- 4. DISPLAY QUANTITATIVE ENGINE (ITC) ---
st.subheader("📊 Quantitative Engine (ITC Trade Map)")
col1, col2, col3 = st.columns(3)
col1.metric("Export Volume (2024)", f"{target_data['Pak_Export_Qty']:,.0f} Tons")
col2.metric("Export Value (2024)", f"${target_data['Pak_Export_Value_kUSD']:,.0f} k")
col3.metric("Unit Value (Price)", f"${target_data['Pak_Unit_Value_USD']:,.2f} / Ton")

# --- 5. QUALITATIVE ENGINE (MOCK TIPP/CUSTOMS) ---
st.subheader("⚖️ Qualitative Engine (Regulatory/Logistics)")
mock_tipp_context = st.text_area(
    "Live Compliance & Logistics Shocks (Edit to test Agent reactivity):",
    "1. TORKHAM BORDER: Currently experiencing 4-day delays due to scanning capacity limits.\n"
    "2. NLC CORRIDORS: Active mandate for mandatory tracking on perishable goods to Central Asia.\n"
    "3. CUSTOMS ACT SEC 138: Enhanced scrutiny on under-invoicing for Chapter 07 and 08; E-Form validation requires secondary DPP Phytosanitary certificate.\n"
    "4. GLOBAL PRICE ALERT: UAE market showing 12% premium on Pakistani citrus and onions via Sea freight."
)

# --- 6. AI TRADE AGENT SIGNAL ---
st.subheader("🤖 Agentic Strategy: Buy / Sell / Reroute Signal")

if st.button("Generate Trade Strategy"):
    if not api_key:
        st.warning("Please enter your Gemini API Key in the sidebar.")
    else:
        with st.spinner("Agent is analyzing ITC Data and Regulatory Context..."):
            genai.configure(api_key=api_key)
            model = genai.GenerativeModel('gemini-1.5-pro')

            # The System Prompt (Where the magic happens)
            prompt = f"""
            You are a Senior Agricultural Trade Agent advising a Pakistani exporter.

            COMMODITY DATA (ITC):
            - HS Code: {target_data['HS_Code']}
            - Product: {target_data['Product_Description']}
            - Current Pakistan Unit Value: ${target_data['Pak_Unit_Value_USD']} per Ton

            REGULATORY & LOGISTICS CONTEXT (TIPP/Customs):
            {mock_tipp_context}

            TASK:
            Based on the quantitative data and the qualitative logistics/regulatory shocks, provide a highly professional, actionable strategy.
            Output your response in exactly 3 sections:
            1. SIGNAL: (Choose one: PROCEED, HOLD, or REROUTE)
            2. LOGISTICS DIRECTIVE: (How to move the cargo, mentioning NLC/Torkham/Sea routes based on context).
            3. COMPLIANCE CHECKLIST: (What specific customs/phytosanitary docs are needed based on Sec 138 context).

            Keep it concise, like a World Bank or PHDEC policy brief.
            """

            response = model.generate_content(prompt)
            st.success("Analysis Complete!")
            st.write(response.text)

Writing app.py


In [ ]:
# 1. Install Streamlit and Localtunnel (runs quietly)
!pip install -q streamlit
!npm install -g localtunnel

# 2. Get the "Password" (Endpoint IP) for your secure tunnel
import urllib
print("👉 COPY THIS IP ADDRESS (YOUR PASSWORD):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

# 3. Run the Streamlit app and open the tunnel
!streamlit run app.py & npx localtunnel --port 8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 127.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏👉 COPY THIS IP ADDRESS (YOUR PASSWORD): 34.80.186.48
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.186.48:8501

your url is: https://three-birds-yawn.loca.lt
/tools/node/lib/node_modules/localtunnel/bin/lt.js:81
    throw err;
    ^

Error: connection refused: localtunnel.me:20217 (check your firewall settings)
    at Socket.<anonymous> (/tools/node/lib/node_modules/localtunnel/lib/TunnelCluster.js:52:11)
    at Socket.emit (node:events:524:28)
    at emitErrorNT (node:internal/streams/destroy:169:8)
    at emitErrorCloseNT (node:internal/streams/destroy:128:3)
    at process.p

In [ ]:
import subprocess
import time
from google.colab import output

# 1. Kill any old crashed streamlit or tunnel processes to free up the port
!pkill -f streamlit
!pkill -f localtunnel

# 2. Start the Streamlit app quietly in the background
print("Starting the MVP server...")
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])

# 3. Give it 3 seconds to fully boot up
time.sleep(3)

# 4. Use Google Colab's native, highly stable secure link
print("✅ Server is running! Click the secure link below to open your dashboard:")
output.serve_kernel_port_as_window(8501)

Starting the MVP server...
✅ Server is running! Click the secure link below to open your dashboard:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
import subprocess
import time

# 1. Kill any stuck processes
!pkill -f streamlit
!pkill -f cloudflared

# 2. Start Streamlit in the background
print("Starting Streamlit...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(3)

# 3. Download Cloudflare Tunnel (Ultra-stable, no passwords needed)
print("Downloading Cloudflare Tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 4. Open the secure tunnel
print("Opening secure tunnel...")
process = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8501'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

time.sleep(4)

# 5. Extract and print the secure link
print("\n✅ SERVER IS READY! Click the link below to open your MVP:")
print("-" * 50)
for line in iter(process.stdout.readline, b''):
    line_str = line.decode('utf-8')
    if "trycloudflare.com" in line_str:
        # Isolate just the URL
        url = line_str[line_str.find("https://"):].strip()
        print(f"👉 {url}")
        print("-" * 50)
        break

Starting Streamlit...
Opening secure tunnel...

✅ SERVER IS READY! Click the link below to open your MVP:
--------------------------------------------------
👉 
--------------------------------------------------


In [ ]:
import subprocess
import time

# 1. Kill any stuck processes from previous attempts
!pkill -f streamlit
!pkill -f cloudflared
!pkill -f ssh

# 2. Start your Streamlit MVP in the background
print("Starting Pakistan Agri-Trade Agent (Streamlit)...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])

# Give it a few seconds to boot up
time.sleep(4)

# 3. Create the Pinggy Tunnel (This will output a big box with your link)
print("✅ Opening Pinggy Tunnel. Look for the 'https://....pinggy.link' URL below!")
print("-" * 60)
!ssh -p 443 -R0:localhost:8501 -o StrictHostKeyChecking=no -o ServerAliveInterval=30 a.pinggy.io

Starting Pakistan Agri-Trade Agent (Streamlit)...
✅ Opening Pinggy Tunnel. Look for the 'https://....pinggy.link' URL below!
------------------------------------------------------------
Allocated port 7 for remote forward to localhost:8501
7=)0]8;;\                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ┌────────────────────────────┐                                                  │

In [2]:
# 1. Install Gradio
!pip install -q gradio

import gradio as gr
import pandas as pd
import google.generativeai as genai

# --- 2. LOAD QUANTITATIVE DATA (ITC) ---
def load_data():
    def load_and_clean(file_path, chapter_name):
        try:
            df = pd.read_csv(file_path, sep='\t', dtype={'﻿Product Code': str})
            df.rename(columns={df.columns[0]: 'HS_Code'}, inplace=True)
            col_map = {
                "Product Label": "Product_Description",
                "Select your indicators-Value in 2024, USD thousand": "Pak_Export_Value_kUSD",
                "Select your indicators-Quantity exported in 2024": "Pak_Export_Qty",
                "Select your indicators-Unit value (USD/unit)": "Pak_Unit_Value_USD"
            }
            df = df.rename(columns=col_map)
            cols_to_keep = ['HS_Code', 'Product_Description', 'Pak_Export_Value_kUSD', 'Pak_Export_Qty', 'Pak_Unit_Value_USD']
            df = df[[col for col in cols_to_keep if col in df.columns]]
            df.insert(1, 'Chapter', chapter_name)
            df.replace('No quantity', 0, inplace=True)
            df.fillna(0, inplace=True)
            for col in ['Pak_Export_Value_kUSD', 'Pak_Export_Qty', 'Pak_Unit_Value_USD']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            return df
        except FileNotFoundError:
            return pd.DataFrame()

    ch06 = load_and_clean('Trade_Map_-chapter_6.txt', 'Ch 06')
    ch07 = load_and_clean('Trade_Map_Chapter_7_2024.txt', 'Ch 07')
    ch08 = load_and_clean('Trade_Map_Chapter_8_2024.txt', 'Ch 08')
    ch20 = load_and_clean('Trade_Map_-Chapter_20_2024.txt', 'Ch 20')

    return pd.concat([ch06, ch07, ch08, ch20], ignore_index=True)

df = load_data()
if not df.empty:
    df['Display_Name'] = df['HS_Code'].astype(str) + " - " + df['Product_Description'].str[:40] + "..."
    commodity_choices = df['Display_Name'].tolist()
else:
    commodity_choices = ["Data not found!"]

# --- 3. AGENT LOGIC ---
def get_trade_strategy(api_key, selected_item, mock_context):
    if not api_key:
        return "⚠️ Error: Please enter your Gemini API Key."
    if selected_item == "Data not found!":
        return "⚠️ Error: ITC Data not loaded."

    target_data = df[df['Display_Name'] == selected_item].iloc[0]

    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-1.5-pro')

        prompt = f"""
        You are a Senior Agricultural Trade Agent advising a Pakistani exporter.

        COMMODITY DATA (ITC):
        - HS Code: {target_data['HS_Code']}
        - Product: {target_data['Product_Description']}
        - Export Volume (2024): {target_data['Pak_Export_Qty']} Tons
        - Export Value (2024): ${target_data['Pak_Export_Value_kUSD']}k
        - Current Pakistan Unit Value: ${target_data['Pak_Unit_Value_USD']} per Ton

        REGULATORY & LOGISTICS CONTEXT (TIPP/Customs):
        {mock_context}

        TASK:
        Based on the quantitative data and the qualitative logistics/regulatory shocks, provide a highly professional, actionable strategy.
        Output your response in exactly 3 sections:
        1. SIGNAL: (Choose one: PROCEED, HOLD, or REROUTE)
        2. LOGISTICS DIRECTIVE: (How to move the cargo, mentioning NLC/Torkham/Sea routes based on context).
        3. COMPLIANCE CHECKLIST: (What specific customs/phytosanitary docs are needed based on Sec 138 context).

        Keep it concise, like a World Bank or PHDEC policy brief.
        """

        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"⚠️ API Error: {str(e)}"

# --- 4. GRADIO DASHBOARD UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇵🇰 Pakistan Agri-Trade AI Agent (MVP)")
    gr.Markdown("**Powered by ITC Trade Map & Gemini 1.5 Pro (World Bank/PHDEC Framework)**")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Configuration")
            api_key_input = gr.Textbox(label="Gemini API Key", type="password", placeholder="Paste your API key here...")
            commodity_dropdown = gr.Dropdown(choices=commodity_choices, label="Select Commodity (HS Code)", value=commodity_choices[0] if commodity_choices else None)

            gr.Markdown("### ⚖️ Qualitative Engine (Regulatory Shocks)")
            context_input = gr.Textbox(
                label="Live Compliance & Logistics Shocks (Edit to test reactivity)",
                lines=7,
                value="1. TORKHAM BORDER: Currently experiencing 4-day delays due to scanning capacity limits.\n2. NLC CORRIDORS: Active mandate for mandatory tracking on perishable goods to Central Asia.\n3. CUSTOMS ACT SEC 138: Enhanced scrutiny on under-invoicing for Chapter 07 and 08; E-Form validation requires secondary DPP Phytosanitary certificate.\n4. GLOBAL PRICE ALERT: UAE market showing 12% premium on Pakistani citrus and onions via Sea freight."
            )
            submit_btn = gr.Button("🚀 Generate Trade Strategy", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 🤖 AI Agentic Strategy Output")
            output_display = gr.Markdown("Enter your API key, select a commodity, and click Generate to see the strategy here.")

    submit_btn.click(
        fn=get_trade_strategy,
        inputs=[api_key_input, commodity_dropdown, context_input],
        outputs=output_display
    )

# Launch with share=True to generate a clean public link!
demo.launch(share=True, debug=True)

/tmp/ipython-input-2571202621.py:88: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://49a7889920e482104e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://49a7889920e482104e.gradio.live
